# 05 — Same abstract graph, certified distinct spatial embeddings

Fix one trivalent theta graph (\(V=2,E=3\)). Tie local knots into one edge, certify inequivalence independently from the HOMFLY-PT fingerprints of the three constituent cycles (Topoly), then compute KnottedGraph's Yamada polynomial. Yamada is never used to construct or certify the examples.


In [ ]:
from __future__ import annotations
from pathlib import Path
import re, sys
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import sympy as sp

ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/"pyproject.toml").exists(): ROOT=ROOT.parent
SRC=ROOT/"src"
if not (SRC/"knotted_graph").exists(): raise RuntimeError("Run inside the KnottedGraph checkout.")
if str(SRC) not in sys.path: sys.path.insert(0,str(SRC))

import knotted_graph, topoly
from knotted_graph.projection import compute_yamada_polynomial
from topoly import homfly
from topoly.params import Closure
if SRC not in Path(knotted_graph.__file__).resolve().parents:
    raise RuntimeError("Imported a stale knotted_graph instead of this checkout.")
A=sp.Symbol("A")
print("KnottedGraph:",Path(knotted_graph.__file__).resolve())
print("Topoly:",Path(topoly.__file__).resolve())


In [ ]:
U=np.array([-1.1,0.,0.]); V=np.array([1.1,0.,0.])

def base_theta(n=500):
    t=np.linspace(0,1,n); x=U[0]+(V[0]-U[0])*t
    curves={
        "e1":np.c_[x,0*t,0*t],
        "e2":np.c_[x,.92*np.sin(np.pi*t), .08*np.sin(2*np.pi*t)],
        "e3":np.c_[x,-.92*np.sin(np.pi*t),-.08*np.sin(2*np.pi*t)],
    }
    G=nx.MultiGraph(); G.add_node("u",pos=U.copy()); G.add_node("v",pos=V.copy())
    for role,P in curves.items(): G.add_edge("u","v",role=role,pts=P)
    return G

def norm(P):
    P=np.asarray(P,float); P-=P.mean(0); return P/np.max(np.linalg.norm(P,axis=1))

def torus(p,q,n=1400):
    t=np.linspace(0,2*np.pi,n,endpoint=False); r=1+.38*np.cos(q*t)
    return norm(np.c_[r*np.cos(p*t),r*np.sin(p*t),.38*np.sin(q*t)])

def figure8(n=1600):
    t=np.linspace(0,2*np.pi,n,endpoint=False)
    return norm(np.c_[(2+np.cos(2*t))*np.cos(3*t),(2+np.cos(2*t))*np.sin(3*t),np.sin(4*t)])

TEMPLATES={"3_1":torus(2,3),"4_1":figure8(),"5_1":torus(2,5),"7_1":torus(2,7)}

def fp(P):
    P=np.asarray(P,float)
    if np.allclose(P[0],P[-1]): P=P[:-1]
    out=homfly(P.tolist(),closure=Closure.CLOSED,tries=1,translate=False,
               poly_reduce=True,chiral=False,minimal=False,hide_trivial=False,
               max_cross=25,cuda=False,run_parallel=False)
    return re.sub(r"\s+","",str(out))

t=np.linspace(0,2*np.pi,700,endpoint=False)
UNKNOT_FP=fp(np.c_[np.cos(t),np.sin(t),0*t])
PRIME_FP={k:fp(P) for k,P in TEMPLATES.items()}
assert all(v!=UNKNOT_FP for v in PRIME_FP.values())
assert len(set(PRIME_FP.values()))==len(PRIME_FP)
print("PASS: prime templates are nontrivial and pairwise HOMFLY-distinct.")


In [ ]:
def basis(a,b):
    ex=(b-a)/np.linalg.norm(b-a); ref=np.array([0.,0.,1.])
    if abs(ex@ref)>.9: ref=np.array([0.,1.,0.])
    ey=np.cross(ref,ex); ey/=np.linalg.norm(ey); ez=np.cross(ex,ey)
    return np.vstack([ex,ey,ez])

def complementary(P,start,gap):
    n=len(P); end=(start+gap)%n
    return np.vstack([P[end:],P[:start+1]]) if start<end else P[end:start+1]

def open_template(name,P):
    candidates=[]; n=len(P)
    for frac in (.025,.04,.06,.085,.12,.16,.22):
        gap=max(4,int(round(frac*n)))
        for start in np.linspace(0,n-1,28,dtype=int):
            Q=complementary(P,int(start),gap)
            if len(Q)<20: continue
            a,b=Q[0],Q[-1]
            if np.linalg.norm(b-a)<1e-8: continue
            B=basis(a,b); X=(Q-.5*(a+b))@B.T; chord=np.linalg.norm(b-a)
            long=np.max(np.abs(X[:,0]))/(.5*chord)
            trans=np.max(np.linalg.norm(X[:,1:],axis=1))/chord
            candidates.append((long+.35*trans,long,Q))
    for _,long,Q in sorted(candidates,key=lambda x:x[0])[:80]:
        if long<=2 and fp(Q)==PRIME_FP[name]: return Q
    raise RuntimeError(f"No straight-closure-certified open arc found for {name}.")

OPEN={k:open_template(k,P) for k,P in TEMPLATES.items()}
print("PASS: all open local-knot templates preserve their closed HOMFLY fingerprint.")


In [ ]:
def map_arc(Q,left,right,radius):
    left=np.asarray(left,float); right=np.asarray(right,float); mid=.5*(left+right)
    a,b=Q[0],Q[-1]; X=(Q-.5*(a+b))@basis(a,b).T
    L=np.linalg.norm(right-left); X[:,0]*=L/np.linalg.norm(b-a)
    tr=np.max(np.linalg.norm(X[:,1:],axis=1)); X[:,1:]*=radius/tr
    ex=(right-left)/L; ref=np.array([0.,0.,1.])
    if abs(ex@ref)>.9: ref=np.array([0.,1.,0.])
    ey=np.cross(ref,ex); ey/=np.linalg.norm(ey); ez=np.cross(ex,ey)
    P=X@np.vstack([ex,ey,ez])+mid; P[0]=left; P[-1]=right
    return P

def knotted_e1(factors):
    if not factors: return np.linspace(U,V,600)
    slots=[(-.47,.47,.25)] if len(factors)==1 else [(-.82,-.08,.15),(.08,.82,.15)]
    pieces=[]; cursor=U.copy()
    for factor,(xa,xb,r) in zip(factors,slots):
        left=np.array([xa,0.,0.]); right=np.array([xb,0.,0.])
        pieces.append(np.linspace(cursor,left,60,endpoint=False))
        pieces.append(map_arc(OPEN[factor],left,right,r)[:-1]); cursor=right
    pieces.append(np.linspace(cursor,V,80))
    P=np.vstack(pieces); P[0]=U; P[-1]=V; return P

def make_theta(factors):
    G=base_theta()
    for u,v,k,d in G.edges(keys=True,data=True):
        if d["role"]=="e1": G[u][v][k]["pts"]=knotted_e1(factors); break
    assert G.number_of_nodes()==2 and G.number_of_edges()==3
    assert sorted(dict(G.degree()).values())==[3,3]
    return G

VARIANTS={
 "0_1":(), "3_1":("3_1",), "4_1":("4_1",), "5_1":("5_1",), "7_1":("7_1",),
 "3_1#3_1":("3_1","3_1"), "3_1#4_1":("3_1","4_1"),
 "3_1#5_1":("3_1","5_1"), "4_1#4_1":("4_1","4_1"), "4_1#5_1":("4_1","5_1"),
}
EMBEDDINGS={name:make_theta(factors) for name,factors in VARIANTS.items()}
print("PASS: 10 candidates have identical theta combinatorics.")


In [ ]:
def edge(G,role):
    for _,_,_,d in G.edges(keys=True,data=True):
        if d["role"]==role: return np.asarray(d["pts"],float)
    raise KeyError(role)

def cycle(P,Q):
    C=np.vstack([P,Q[-2::-1]])
    return C if np.allclose(C[0],C[-1]) else np.vstack([C,C[0]])

def signature(G):
    e1,e2,e3=edge(G,"e1"),edge(G,"e2"),edge(G,"e3")
    f12,f13,f23=fp(cycle(e1,e2)),fp(cycle(e1,e3)),fp(cycle(e2,e3))
    if f23!=UNKNOT_FP: raise RuntimeError("Untouched theta cycle became nontrivial.")
    if f12!=f13: raise RuntimeError("The two local-knot-containing cycles disagree.")
    return tuple(sorted((f12,f13,f23)))

SIGNATURES={name:signature(G) for name,G in EMBEDDINGS.items()}
if len(set(SIGNATURES.values()))!=10:
    groups={}
    for name,s in SIGNATURES.items(): groups.setdefault(s,[]).append(name)
    raise RuntimeError("Independent HOMFLY certificate has duplicates: "+
                       repr([g for g in groups.values() if len(g)>1]))
print("PASS: all 10 theta embeddings are pairwise certified inequivalent independently of Yamada.")


In [ ]:
def yamada(G):
    r=compute_yamada_polynomial(G,A,rotation_angles=None,num_rotation_samples=18,
        normalize=True,n_jobs=1,method="recursive",return_result=True)
    return sp.expand(r.polynomial),int(r.projection.num_crossings)

def same(a,b): return sp.simplify(sp.together(sp.expand(a-b)))==0

YAMADA={}; CROSSINGS={}
for name,G in EMBEDDINGS.items():
    YAMADA[name],CROSSINGS[name]=yamada(G)
    print(f"{name:10s} crossings={CROSSINGS[name]:2d}  Yamada={YAMADA[name]}")

groups=[]; unused=set(YAMADA)
while unused:
    seed=sorted(unused)[0]
    g=[x for x in sorted(unused) if same(YAMADA[seed],YAMADA[x])]
    groups.append(g); unused.difference_update(g)

print("Certified inequivalent embeddings:",len(EMBEDDINGS))
print("Distinct Yamada polynomials:",len(groups))
print("Discrimination fraction:",len(groups)/len(EMBEDDINGS))
print("Yamada equivalence groups:",groups)

names=list(EMBEDDINGS); M=np.zeros((10,10),int)
for i,a in enumerate(names):
    for j,b in enumerate(names): M[i,j]=int(not same(YAMADA[a],YAMADA[b]))
fig,ax=plt.subplots(figsize=(7,6)); im=ax.imshow(M,vmin=0,vmax=1)
ax.set_xticks(range(10),names,rotation=60,ha="right"); ax.set_yticks(range(10),names)
ax.set_title("Yamada pairwise discrimination: fixed theta graph")
fig.colorbar(im,ax=ax,label="1 = distinguished"); plt.tight_layout(); plt.show()


## Hard-case extension

The local-knot suite is the controlled Level-1 benchmark. A future Level-2 section should use trusted literature-backed data for harder theta curves (for example a Kinoshita-type theta curve) where constituent knots alone do not certify the distinction. No such coordinates are fabricated here.
